# DDIM affine: estimate latent map → undo on image → re-invert

Follow-on to `ddim_affine_sign_registration_n5.ipynb` / the latent affine sign-template
experiments. Those recover the latent affine accurately, but **latent** inverse-warp of
`z_hat` only reaches ~60–70% sign-bit accuracy under nontrivial warps.

This notebook tests whether applying the **estimated inverse in pixel space** on the
warped image, then **DDIM-inverting again**, recovers a higher bit accuracy:

1. Generate 5 SD1.5 images from known Gaussian `z` (or reuse cached n5 artifacts).
2. Apply the same pixel-space affines; DDIM-invert → `z_hat`.
3. Sign-template latent affine registration (matched baseline bits vs full observation).
4. Convert the estimate to pixel units (`t × 8`), apply that inverse (and an oracle true
   inverse) to the warped image, save undos, and DDIM-invert again.
5. Compare bit accuracy of: raw warped invert, latent inverse-warp (~60% baseline),
   image-undo+re-invert (estimated), and image-undo+re-invert (oracle).

### HiPerGator (OOD Jupyter)

- Kernel: `gaussian-rotation`
- **One GPU**, ~8 GB, account/QoS `eel5840`, partition `hpg-turin`
- Working directory: **repo root**
- Prefer `REUSE_FROM = results/ddim_affine_sign_registration_n5` when that run is complete
  (skips generation + first invert). Otherwise generates into this notebook's output dir.
- Extra GPU work: 5 samples × 4 non-baseline × 2 (est+oracle) ≈ 40 inversions.
- Re-running skips samples whose undo keys are already present.



## 1. Configuration



In [ ]:
from __future__ import annotations

import json
import shutil
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy import ndimage
from scipy.optimize import minimize
from scipy.stats import pearsonr

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

NUM_SAMPLES = 5
SEED = 42
REFERENCE_SAMPLE_COUNT = 100
OUTPUT_DIR = Path("results/ddim_affine_sign_image_undo_n5")
# If this prior run is complete, reuse its images + first-pass latents.
REUSE_FROM = Path("results/ddim_affine_sign_registration_n5")
LATENT_DIR = OUTPUT_DIR / "latents"
IMAGE_DIR = OUTPUT_DIR / "images"
FIGURES_DIR = OUTPUT_DIR / "figures"

MODEL_ID = "runwayml/stable-diffusion-v1-5"
NUM_INFERENCE_STEPS = 50
DEVICE = "cuda"
DTYPE = "float16"
IMAGE_SIZE = 512
PROMPT = "a photograph of an astronaut riding a horse"
GUIDANCE_SCALE = 7.5
VAE_SCALE_FACTOR = 8

# Same pixel-space affine set as ddim_affine_sign_registration_n5.ipynb.
# params = (angle_deg, scale_x, scale_y, shear, tx_px, ty_px)
CONDITIONS = [
    {"name": "baseline", "params": (0.0, 1.0, 1.0, 0.0, 0.0, 0.0)},
    {"name": "rotation_10", "params": (10.0, 1.0, 1.0, 0.0, 0.0, 0.0)},
    {"name": "hshift_10", "params": (0.0, 1.0, 1.0, 0.0, 10.0, 0.0)},
    {"name": "vshift_10", "params": (0.0, 1.0, 1.0, 0.0, 0.0, 10.0)},
    {"name": "combined_mild", "params": (8.0, 1.05, 0.98, 0.05, 6.0, -4.0)},
]
NONBASELINE = [c for c in CONDITIONS if c["name"] != "baseline"]

PARAM_NAMES = ["angle_deg", "scale_x", "scale_y", "shear", "tx", "ty"]
ANGLE_RANGE = (-20.0, 20.0)
SCALE_RANGE = (0.85, 1.15)
SHEAR_RANGE = (-0.2, 0.2)
TRANSLATION_RANGE = (-4.0, 4.0)
BOUNDS = [ANGLE_RANGE, SCALE_RANGE, SCALE_RANGE, SHEAR_RANGE, TRANSLATION_RANGE, TRANSLATION_RANGE]
LB = np.array([b[0] for b in BOUNDS], dtype=float)
UB = np.array([b[1] for b in BOUNDS], dtype=float)
SCIPY_BOUNDS = list(zip(LB, UB))

WARP_ORDER = 1
VALID_RADIUS_FRAC = 0.32
COARSE_ANGLE_STEP = 4.0
COARSE_TRANSLATION_STEP = 1.0

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})
for path in (OUTPUT_DIR, LATENT_DIR, IMAGE_DIR, FIGURES_DIR):
    path.mkdir(parents=True, exist_ok=True)

n_extra_inv = NUM_SAMPLES * len(NONBASELINE) * 2
print(f"samples={NUM_SAMPLES}, conditions={len(CONDITIONS)}, nonbaseline={len(NONBASELINE)}")
print(f"extra image-undo inversions (est+oracle): {n_extra_inv}")
print(f"reuse_from -> {REUSE_FROM.resolve()}")
print(f"output -> {OUTPUT_DIR.resolve()}")


## 2. Geometry helpers

Same six-parameter affine and sign-template registration stack as
`ddim_affine_sign_registration_n5.ipynb`. Latent estimates convert to pixel warps by
scaling translations by the VAE factor (8). Image undos use the matrix form of the
inverse so we do not need to re-parameterize `A^{-1}`.



In [ ]:
def fmt_duration(seconds: float) -> str:
    seconds = max(0.0, float(seconds))
    if seconds < 60:
        return f"{seconds:.0f}s"
    minutes, secs = divmod(int(round(seconds)), 60)
    if minutes < 60:
        return f"{minutes}m{secs:02d}s"
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h{minutes:02d}m"


class StepClock:
    def __init__(self, total: int, label: str):
        self.total = total
        self.label = label
        self.done = 0
        self.started = time.perf_counter()
        self.durations: list[float] = []
        self._tick = self.started

    def begin(self) -> None:
        self._tick = time.perf_counter()

    def end(self, detail: str) -> None:
        now = time.perf_counter()
        duration = now - self._tick
        self.durations.append(duration)
        self.done += 1
        average = sum(self.durations) / len(self.durations)
        remaining = self.total - self.done
        eta = average * remaining
        print(
            f"[{self.label} {self.done}/{self.total}] {detail} | "
            f"this {duration:.1f}s avg {average:.1f}s | "
            f"elapsed {fmt_duration(now - self.started)} | "
            f"~{fmt_duration(eta)} left",
            flush=True,
        )


def sample_seeds() -> list[int]:
    rng = np.random.default_rng(SEED)
    return [int(s) for s in rng.integers(0, 2**31 - 1, size=REFERENCE_SAMPLE_COUNT, endpoint=False)[:NUM_SAMPLES]]


def latent_key(name: str) -> str:
    return f"z_hat__{name}"


def undo_latent_key(name: str, arm: str) -> str:
    return f"z_hat__{name}__image_undo_{arm}"


def sample_path(sample_id: int) -> Path:
    return LATENT_DIR / f"sample_{sample_id:03d}.pt"


def sample_image_dir(sample_id: int) -> Path:
    path = IMAGE_DIR / f"sample_{sample_id:03d}"
    path.mkdir(parents=True, exist_ok=True)
    return path


def first_pass_complete(sample_id: int, latent_dir: Path = LATENT_DIR) -> bool:
    path = latent_dir / f"sample_{sample_id:03d}.pt"
    if not path.is_file():
        return False
    blob = torch.load(path, map_location="cpu", weights_only=True)
    needed = {"z", "image_seed"} | {latent_key(c["name"]) for c in CONDITIONS}
    return needed.issubset(blob.keys())


def undo_pass_complete(sample_id: int) -> bool:
    path = sample_path(sample_id)
    if not path.is_file():
        return False
    blob = torch.load(path, map_location="cpu", weights_only=True)
    needed = {undo_latent_key(c["name"], arm) for c in NONBASELINE for arm in ("est", "oracle")}
    return needed.issubset(blob.keys())


def params_to_matrix(params) -> tuple[np.ndarray, np.ndarray]:
    angle, sx, sy, shear, tx, ty = np.asarray(params, dtype=float)
    theta = np.deg2rad(angle)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    rotation = np.array([[cos_t, -sin_t], [sin_t, cos_t]], dtype=float)
    linear = np.array([[sx, sx * shear], [0.0, sy]], dtype=float)
    return rotation @ linear, np.array([tx, ty], dtype=float)


def image_center(spatial_shape: tuple[int, int]) -> np.ndarray:
    return np.array(spatial_shape, dtype=float) / 2.0 - 0.5


def warp_array(image: np.ndarray, matrix: np.ndarray, translation: np.ndarray) -> np.ndarray:
    if image.ndim == 3 and image.shape[0] in (1, 3, 4) and image.shape[-1] not in (1, 3):
        channels = image
        spatial = image.shape[-2:]
        stack_axis = 0
    elif image.ndim == 3:
        channels = np.moveaxis(image, -1, 0)
        spatial = image.shape[:2]
        stack_axis = -1
    elif image.ndim == 2:
        channels = image[None]
        spatial = image.shape
        stack_axis = 0
    else:
        raise ValueError(f"unsupported image shape {image.shape}")

    center = image_center(spatial)
    inverse = np.linalg.inv(matrix)
    affine = np.zeros((2, 3), dtype=float)
    affine[:, :2] = inverse
    affine[:, 2] = -inverse @ (center + translation) + center
    warped = np.stack(
        [
            ndimage.affine_transform(
                channel,
                affine,
                order=WARP_ORDER,
                mode="constant",
                cval=0.0,
                prefilter=True,
            )
            for channel in channels
        ],
        axis=0,
    )
    if stack_axis == -1:
        return np.moveaxis(warped, 0, -1)
    if image.ndim == 2:
        return warped[0]
    return warped


def warp_pil(image: Image.Image, params) -> Image.Image:
    rgb = np.asarray(image.convert("RGB"), dtype=np.float64) / 255.0
    warped = np.clip(warp_array(rgb, *params_to_matrix(params)), 0.0, 1.0)
    return Image.fromarray((warped * 255.0).round().astype(np.uint8), mode="RGB")


def warp_pil_matrix(image: Image.Image, matrix: np.ndarray, translation: np.ndarray) -> Image.Image:
    rgb = np.asarray(image.convert("RGB"), dtype=np.float64) / 255.0
    warped = np.clip(warp_array(rgb, matrix, translation), 0.0, 1.0)
    return Image.fromarray((warped * 255.0).round().astype(np.uint8), mode="RGB")


def expected_latent_params(pixel_params) -> np.ndarray:
    angle, sx, sy, shear, tx, ty = np.asarray(pixel_params, dtype=float)
    return np.array(
        [angle, sx, sy, shear, tx / VAE_SCALE_FACTOR, ty / VAE_SCALE_FACTOR],
        dtype=float,
    )


def latent_to_pixel_params(latent_params) -> np.ndarray:
    angle, sx, sy, shear, tx, ty = np.asarray(latent_params, dtype=float)
    return np.array(
        [angle, sx, sy, shear, tx * VAE_SCALE_FACTOR, ty * VAE_SCALE_FACTOR],
        dtype=float,
    )


def as_chw(tensor) -> np.ndarray:
    array = np.asarray(tensor, dtype=np.float64)
    if array.ndim == 4 and array.shape[0] == 1:
        array = array[0]
    if array.ndim != 3:
        raise ValueError(f"expected (C,H,W), got {array.shape}")
    return array


def invert_params(params: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    matrix, translation = params_to_matrix(params)
    inverse = np.linalg.inv(matrix)
    return inverse, -inverse @ translation


def apply_pixel_inverse(image: Image.Image, forward_pixel_params) -> Image.Image:
    """Undo a forward pixel affine by warping with its inverse (A^{-1}, -A^{-1}t)."""
    return warp_pil_matrix(image, *invert_params(np.asarray(forward_pixel_params, dtype=float)))


def circular_valid_mask(spatial_shape: tuple[int, int], radius_frac: float = VALID_RADIUS_FRAC) -> np.ndarray:
    height, width = spatial_shape
    center_y, center_x = image_center(spatial_shape)
    rows, cols = np.mgrid[0:height, 0:width]
    radius = radius_frac * float(min(height, width))
    return (rows - center_y) ** 2 + (cols - center_x) ** 2 <= radius ** 2


def bits_from_latent(latent: np.ndarray) -> np.ndarray:
    return (np.asarray(latent) >= 0).astype(np.float64)


def bits_to_pm(bits01: np.ndarray) -> np.ndarray:
    return 2.0 * bits01 - 1.0


def sign_pm(latent: np.ndarray) -> np.ndarray:
    return np.where(np.asarray(latent) >= 0, 1.0, -1.0)


def masked_vectors(a: np.ndarray, b: np.ndarray, mask: np.ndarray):
    return a[:, mask].ravel(), b[:, mask].ravel()


def full_latent_corr(a: np.ndarray, b: np.ndarray, mask: np.ndarray | None = None) -> float:
    if mask is None:
        left, right = a.ravel(), b.ravel()
    else:
        left, right = masked_vectors(a, b, mask)
    if left.size < 2 or np.std(left) < 1e-12 or np.std(right) < 1e-12:
        return 0.0
    return float(pearsonr(left, right).statistic)


def bit_metrics(bits01: np.ndarray, observation: np.ndarray, params: np.ndarray, mask: np.ndarray) -> dict:
    bits_pm = bits_to_pm(bits01)
    forward = warp_array(bits_pm, *params_to_matrix(params))
    back = warp_array(observation, *invert_params(params))
    signed_back = sign_pm(back)
    bits_m, signed_m = masked_vectors(bits_pm, signed_back, mask)
    _, back_m = masked_vectors(bits_pm, back, mask)
    return {
        "forward_correlation": full_latent_corr(forward, observation, mask),
        "bit_accuracy": float(np.mean(bits_m == signed_m)),
        "bit_mse": float(np.mean((bits_m - signed_m) ** 2)),
        "bit_correlation": full_latent_corr(bits_pm, signed_back, mask),
        "continuous_correlation": full_latent_corr(bits_pm, back, mask),
        "continuous_mse": float(np.mean((bits_m - back_m) ** 2)),
    }


def sign_bit_accuracy(reference: np.ndarray, other: np.ndarray, mask: np.ndarray) -> float:
    left, right = masked_vectors(sign_pm(reference), sign_pm(other), mask)
    return float(np.mean(left == right))


print("conditions:")
for cond in CONDITIONS:
    exp = expected_latent_params(cond["params"])
    print(f"  {cond['name']}: pixel={tuple(cond['params'])} -> latent tx/ty=({exp[4]:.3f},{exp[5]:.3f})")


In [ ]:
def forward_correlation(bits_pm: np.ndarray, observation: np.ndarray, params: np.ndarray, mask: np.ndarray) -> float:
    warped = warp_array(bits_pm, *params_to_matrix(params))
    return full_latent_corr(warped, observation, mask)


def coarse_initialize(bits_pm: np.ndarray, observation: np.ndarray, mask: np.ndarray) -> np.ndarray:
    identity = np.array([0.0, 1.0, 1.0, 0.0, 0.0, 0.0], dtype=float)
    best_params = identity.copy()
    best_corr = forward_correlation(bits_pm, observation, best_params, mask)

    for angle in np.arange(ANGLE_RANGE[0], ANGLE_RANGE[1] + 0.5 * COARSE_ANGLE_STEP, COARSE_ANGLE_STEP):
        params = np.array([angle, 1.0, 1.0, 0.0, 0.0, 0.0], dtype=float)
        corr = forward_correlation(bits_pm, observation, params, mask)
        if corr > best_corr:
            best_corr, best_params = corr, params

    for tx in np.arange(TRANSLATION_RANGE[0], TRANSLATION_RANGE[1] + 0.5 * COARSE_TRANSLATION_STEP, COARSE_TRANSLATION_STEP):
        for ty in np.arange(TRANSLATION_RANGE[0], TRANSLATION_RANGE[1] + 0.5 * COARSE_TRANSLATION_STEP, COARSE_TRANSLATION_STEP):
            params = np.array([0.0, 1.0, 1.0, 0.0, tx, ty], dtype=float)
            corr = forward_correlation(bits_pm, observation, params, mask)
            if corr > best_corr:
                best_corr, best_params = corr, params

    angle0, _, _, _, tx0, ty0 = best_params
    for angle in np.unique(np.clip([angle0 - COARSE_ANGLE_STEP, angle0, angle0 + COARSE_ANGLE_STEP], *ANGLE_RANGE)):
        for tx in np.unique(np.clip([tx0 - COARSE_TRANSLATION_STEP, tx0, tx0 + COARSE_TRANSLATION_STEP], *TRANSLATION_RANGE)):
            for ty in np.unique(np.clip([ty0 - COARSE_TRANSLATION_STEP, ty0, ty0 + COARSE_TRANSLATION_STEP], *TRANSLATION_RANGE)):
                params = np.array([angle, 1.0, 1.0, 0.0, tx, ty], dtype=float)
                corr = forward_correlation(bits_pm, observation, params, mask)
                if corr > best_corr:
                    best_corr, best_params = corr, params
    return best_params


def register_sign_affine(bits01: np.ndarray, observation: np.ndarray, mask: np.ndarray) -> dict:
    started = time.perf_counter()
    bits_pm = bits_to_pm(bits01)
    init = coarse_initialize(bits_pm, observation, mask)

    def objective(params: np.ndarray) -> float:
        return -forward_correlation(bits_pm, observation, params, mask)

    refined = minimize(
        objective,
        init,
        method="L-BFGS-B",
        bounds=SCIPY_BOUNDS,
        options={"ftol": 1e-9, "gtol": 1e-7, "maxiter": 200},
    )
    estimate = np.asarray(refined.x, dtype=float)
    metrics = bit_metrics(bits01, observation, estimate, mask)
    return {
        "estimated_params": estimate,
        "method": "coarse+L-BFGS-B",
        "runtime_s": time.perf_counter() - started,
        **metrics,
    }


print("registration helpers ready")


## 3. Reuse or generate first-pass images + inversions

If `REUSE_FROM` has complete first-pass `.pt` files and warped PNGs, copy them into
`OUTPUT_DIR`. Otherwise generate from scratch (same routine as the n5 registration
notebook).



In [ ]:
def import_reuse_artifacts() -> int:
    """Copy complete first-pass samples from REUSE_FROM. Returns how many were imported."""
    if not REUSE_FROM.is_dir():
        print(f"REUSE_FROM missing: {REUSE_FROM}")
        return 0
    imported = 0
    for sample_id in range(NUM_SAMPLES):
        if first_pass_complete(sample_id):
            continue
        if not first_pass_complete(sample_id, REUSE_FROM / "latents"):
            continue
        src_pt = REUSE_FROM / "latents" / f"sample_{sample_id:03d}.pt"
        dst_pt = sample_path(sample_id)
        shutil.copy2(src_pt, dst_pt)
        src_img = REUSE_FROM / "images" / f"sample_{sample_id:03d}"
        dst_img = sample_image_dir(sample_id)
        if src_img.is_dir():
            for png in src_img.glob("*.png"):
                shutil.copy2(png, dst_img / png.name)
        imported += 1
        print(f"imported sample {sample_id} from {REUSE_FROM}", flush=True)
    return imported


def run_generate_and_invert() -> None:
    if DEVICE == "cuda" and not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. Use a one-GPU HiPerGator Jupyter session with the gaussian-rotation kernel."
        )
    from experiments.generate_and_invert import (
        _dtype,
        _load_generation_pipeline,
        _load_inversion_pipeline,
        _recovered_latent,
        _release_pipeline,
        _sample_latent,
    )

    seeds = sample_seeds()
    pending = [i for i in range(NUM_SAMPLES) if not first_pass_complete(i)]
    print(
        f"{NUM_SAMPLES - len(pending)}/{NUM_SAMPLES} first-pass samples already complete; "
        f"{len(pending)} still need generation+inversion.",
        flush=True,
    )
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)
    if not pending:
        print("Nothing to invert for first pass.", flush=True)
        return

    dtype = _dtype(DTYPE)
    load_started = time.perf_counter()
    print("Loading generation pipeline...", flush=True)
    pipe = _load_generation_pipeline(MODEL_ID, dtype, DEVICE)
    print(f"Generation ready in {fmt_duration(time.perf_counter() - load_started)}.", flush=True)
    if float(pipe.scheduler.init_noise_sigma) != 1.0:
        raise RuntimeError("DDIM init_noise_sigma is not 1.")
    if pipe.vae_scale_factor != VAE_SCALE_FACTOR:
        raise RuntimeError(f"VAE scale {pipe.vae_scale_factor} != {VAE_SCALE_FACTOR}")

    generated = []
    gen_clock = StepClock(len(pending), "generation")
    try:
        for sample_id in pending:
            gen_clock.begin()
            z = _sample_latent(pipe, seeds[sample_id], IMAGE_SIZE, IMAGE_SIZE, dtype)
            result = pipe(
                prompt=PROMPT,
                latents=z.to(DEVICE),
                num_inference_steps=NUM_INFERENCE_STEPS,
                guidance_scale=GUIDANCE_SCALE,
                height=IMAGE_SIZE,
                width=IMAGE_SIZE,
            )
            if len(result.images) != 1:
                raise RuntimeError(f"sample {sample_id} returned {len(result.images)} images")
            image = result.images[0].convert("RGB")
            if image.size != (IMAGE_SIZE, IMAGE_SIZE):
                raise RuntimeError(f"bad image size {image.size}")
            generated.append((sample_id, z.detach().to(device="cpu", dtype=torch.float32), image))
            gen_clock.end(f"sample {sample_id}, seed {seeds[sample_id]}")
    finally:
        _release_pipeline(pipe)
        print("Generation pipeline released.", flush=True)

    load_started = time.perf_counter()
    print("Loading inversion pipeline...", flush=True)
    inverse = _load_inversion_pipeline(MODEL_ID, dtype, DEVICE)
    print(f"Inversion ready in {fmt_duration(time.perf_counter() - load_started)}.", flush=True)
    if type(inverse.inverse_scheduler).__name__ != "DDIMInverseScheduler":
        raise RuntimeError(f"unexpected inverse scheduler {type(inverse.inverse_scheduler).__name__}")

    inv_clock = StepClock(len(generated) * len(CONDITIONS), "inversion")
    try:
        for sample_id, z, image in generated:
            blob = {"z": z, "image_seed": torch.tensor(seeds[sample_id], dtype=torch.int64)}
            out_img = sample_image_dir(sample_id)
            image.save(out_img / "original.png")
            sample_started = time.perf_counter()
            for cond_index, cond in enumerate(CONDITIONS):
                inv_clock.begin()
                name = cond["name"]
                params = np.asarray(cond["params"], dtype=float)
                torch.manual_seed(SEED + 1000 * sample_id + cond_index)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(SEED + 1000 * sample_id + cond_index)
                warped = warp_pil(image, params) if name != "baseline" else image.copy()
                warped.save(out_img / f"{name}.png")
                inverted = inverse.invert(
                    prompt=PROMPT,
                    image=warped,
                    num_inference_steps=NUM_INFERENCE_STEPS,
                    guidance_scale=GUIDANCE_SCALE,
                    inpaint_strength=1.0,
                    num_reg_steps=0,
                )
                z_hat = _recovered_latent(inverted.latents, z.shape)
                z_hat = z_hat.detach().to(device="cpu", dtype=torch.float32).contiguous()
                if tuple(z_hat.shape) != tuple(z.shape) or not torch.isfinite(z_hat).all():
                    raise RuntimeError(f"bad inverted latent sample {sample_id} {name}")
                blob[latent_key(name)] = z_hat
                inv_clock.end(f"sample {sample_id} {name}")
            torch.save(blob, sample_path(sample_id))
            print(
                f"Saved {sample_path(sample_id)} "
                f"({fmt_duration(time.perf_counter() - sample_started)} for {len(CONDITIONS)} inversions).",
                flush=True,
            )
    finally:
        _release_pipeline(inverse)
        print("Inversion pipeline released.", flush=True)

    from experiments.generate_and_invert import _git_commit as _gc

    meta = {
        "seed": SEED,
        "num_samples": NUM_SAMPLES,
        "sample_seeds": sample_seeds(),
        "samples_first_pass_complete": [i for i in range(NUM_SAMPLES) if first_pass_complete(i)],
        "model_id": MODEL_ID,
        "image_size": IMAGE_SIZE,
        "vae_scale_factor": VAE_SCALE_FACTOR,
        "num_inference_steps": NUM_INFERENCE_STEPS,
        "guidance_scale": GUIDANCE_SCALE,
        "prompt": PROMPT,
        "dtype": DTYPE,
        "conditions": CONDITIONS,
        "reuse_from": str(REUSE_FROM),
        "git_commit": _gc(),
        "note": "pixel affine then DDIM invert; image-undo re-invert in later cells",
    }
    (OUTPUT_DIR / "metadata.json").write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")
    print(f"wrote {OUTPUT_DIR / 'metadata.json'}", flush=True)


run_started = time.perf_counter()
n_imported = import_reuse_artifacts()
print(f"imported {n_imported} samples from reuse dir")
run_generate_and_invert()
print(f"first-pass wall {fmt_duration(time.perf_counter() - run_started)}", flush=True)


## 4. Example warped images (sample 0)



In [ ]:
example_dir = sample_image_dir(0)
names = [c["name"] for c in CONDITIONS]
missing = [n for n in names if not (example_dir / f"{n}.png").is_file()]
if missing:
    raise FileNotFoundError(f"missing example images in {example_dir}: {missing}")

fig, axes = plt.subplots(1, len(names), figsize=(2.6 * len(names), 2.8))
for ax, name in zip(axes, names):
    ax.imshow(Image.open(example_dir / f"{name}.png"))
    ax.set_title(name, fontsize=9)
    ax.axis("off")
fig.suptitle("Sample 0: original / warped images")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "example_warped_images_sample0.png", bbox_inches="tight")
plt.show()


## 5. Sign-template latent affine registration

Matched arm: bits of baseline `z_hat` vs full signed condition observation.
`matched_bit_accuracy` here is the **latent inverse-warp** score (~60% on nontrivial
warps) that the image-undo arm aims to beat.



In [ ]:
incomplete = [i for i in range(NUM_SAMPLES) if not first_pass_complete(i)]
if incomplete:
    raise FileNotFoundError(f"incomplete first-pass samples {incomplete}; finish generate/invert first")

blobs = [torch.load(sample_path(i), map_location="cpu", weights_only=True) for i in range(NUM_SAMPLES)]
SHAPE = as_chw(blobs[0]["z"]).shape
VALID_MASK = circular_valid_mask(SHAPE[-2:])
rng = np.random.default_rng(SEED + 7)
random_bits_by_sample = {i: (rng.random(SHAPE) < 0.5).astype(np.float64) for i in range(NUM_SAMPLES)}

reg_rows = []
reg_started = time.perf_counter()
jobs = [(sid, cond) for sid in range(NUM_SAMPLES) for cond in CONDITIONS]
print(f"registering {len(jobs)} (sample, condition) pairs x matched+random...", flush=True)

estimates: dict[tuple[int, str], np.ndarray] = {}

for index, (sample_id, cond) in enumerate(jobs, start=1):
    blob = blobs[sample_id]
    z = as_chw(blob["z"])
    baseline = as_chw(blob[latent_key("baseline")])
    observation = as_chw(blob[latent_key(cond["name"])])
    expected = expected_latent_params(cond["params"])
    matched_bits = bits_from_latent(baseline)
    random_bits = random_bits_by_sample[sample_id]

    if cond["name"] == "baseline":
        matched = {
            "estimated_params": expected.copy(),
            "method": "identity",
            "runtime_s": 0.0,
            **bit_metrics(matched_bits, observation, expected, VALID_MASK),
        }
        random = {
            "estimated_params": expected.copy(),
            "method": "identity",
            "runtime_s": 0.0,
            **bit_metrics(random_bits, observation, expected, VALID_MASK),
        }
    else:
        matched = register_sign_affine(matched_bits, observation, VALID_MASK)
        random = register_sign_affine(random_bits, observation, VALID_MASK)

    estimates[(sample_id, cond["name"])] = matched["estimated_params"].copy()
    random_at_true = bit_metrics(random_bits, observation, expected, VALID_MASK)
    row = {
        "sample_id": sample_id,
        "condition": cond["name"],
        "matched_method": matched["method"],
        "random_method": random["method"],
        "matched_runtime_s": matched["runtime_s"],
        "random_runtime_s": random["runtime_s"],
        "latent_undo_bit_accuracy": matched["bit_accuracy"],
        "matched_bit_correlation": matched["bit_correlation"],
        "matched_continuous_correlation": matched["continuous_correlation"],
        "matched_forward_correlation": matched["forward_correlation"],
        "random_bit_accuracy": random["bit_accuracy"],
        "random_bit_accuracy_at_true": random_at_true["bit_accuracy"],
        "matched_param_mae": float(np.mean(np.abs(matched["estimated_params"] - expected))),
        "random_param_mae": float(np.mean(np.abs(random["estimated_params"] - expected))),
        "matched_angle_err": float(abs(matched["estimated_params"][0] - expected[0])),
        "matched_tx_err": float(abs(matched["estimated_params"][4] - expected[4])),
        "matched_ty_err": float(abs(matched["estimated_params"][5] - expected[5])),
        "raw_warped_bit_accuracy_vs_baseline": sign_bit_accuracy(baseline, observation, VALID_MASK),
        "raw_warped_bit_accuracy_vs_z": sign_bit_accuracy(z, observation, VALID_MASK),
        "corr_zhat_vs_z": full_latent_corr(observation, z, VALID_MASK),
    }
    for name, truth, pix, matched_hat in zip(
        PARAM_NAMES,
        expected,
        np.asarray(cond["params"], dtype=float),
        matched["estimated_params"],
    ):
        row[f"pixel_{name}"] = float(pix)
        row[f"true_latent_{name}"] = float(truth)
        row[f"matched_est_{name}"] = float(matched_hat)
    reg_rows.append(row)

    if index == 1 or index == len(jobs) or index % max(1, len(jobs) // 5) == 0:
        elapsed = time.perf_counter() - reg_started
        eta = elapsed / index * (len(jobs) - index)
        print(
            f"[{index}/{len(jobs)}] sample {sample_id} {cond['name']} | "
            f"latent_undo bit={row['latent_undo_bit_accuracy']:.3f} "
            f"raw={row['raw_warped_bit_accuracy_vs_baseline']:.3f} | "
            f"ang_err={row['matched_angle_err']:.2f} | "
            f"elapsed {fmt_duration(elapsed)} ~{fmt_duration(eta)} left",
            flush=True,
        )

registration = pd.DataFrame(reg_rows)
registration.to_csv(OUTPUT_DIR / "registration_results.csv", index=False)
print(f"wrote {OUTPUT_DIR / 'registration_results.csv'} ({len(registration)} rows)")
print(f"registration wall {fmt_duration(time.perf_counter() - reg_started)}")


## 6. Image-space undo + re-invert

For each non-baseline condition:

- **est**: inverse of the *estimated* latent affine (translations × 8) applied to the warped PNG
- **oracle**: inverse of the *true* pixel affine

Both undos are DDIM-inverted and stored as `z_hat__{name}__image_undo_{est|oracle}`.



In [ ]:
def run_image_undo_and_reinvert() -> None:
    if DEVICE == "cuda" and not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. Use a one-GPU HiPerGator Jupyter session with the gaussian-rotation kernel."
        )
    from experiments.generate_and_invert import (
        _dtype,
        _load_inversion_pipeline,
        _recovered_latent,
        _release_pipeline,
    )

    pending = [i for i in range(NUM_SAMPLES) if not undo_pass_complete(i)]
    print(
        f"{NUM_SAMPLES - len(pending)}/{NUM_SAMPLES} samples already have image-undo latents; "
        f"{len(pending)} still need re-inversion.",
        flush=True,
    )
    if not pending:
        print("Nothing to re-invert.", flush=True)
        return

    dtype = _dtype(DTYPE)
    load_started = time.perf_counter()
    print("Loading inversion pipeline for image-undo pass...", flush=True)
    inverse = _load_inversion_pipeline(MODEL_ID, dtype, DEVICE)
    print(f"Inversion ready in {fmt_duration(time.perf_counter() - load_started)}.", flush=True)

    n_jobs = len(pending) * len(NONBASELINE) * 2
    inv_clock = StepClock(n_jobs, "image-undo-inv")
    try:
        for sample_id in pending:
            path = sample_path(sample_id)
            blob = torch.load(path, map_location="cpu", weights_only=True)
            z = blob["z"]
            out_img = sample_image_dir(sample_id)
            sample_started = time.perf_counter()
            for cond_index, cond in enumerate(NONBASELINE):
                name = cond["name"]
                warped_path = out_img / f"{name}.png"
                if not warped_path.is_file():
                    raise FileNotFoundError(warped_path)
                warped = Image.open(warped_path).convert("RGB")

                true_pixel = np.asarray(cond["params"], dtype=float)
                est_latent = estimates[(sample_id, name)]
                est_pixel = latent_to_pixel_params(est_latent)

                arms = {
                    "est": apply_pixel_inverse(warped, est_pixel),
                    "oracle": apply_pixel_inverse(warped, true_pixel),
                }
                for arm, undone in arms.items():
                    undone.save(out_img / f"{name}__undo_{arm}.png")
                    key = undo_latent_key(name, arm)
                    if key in blob:
                        inv_clock.begin()
                        inv_clock.end(f"sample {sample_id} {name}/{arm} (cached)")
                        continue
                    inv_clock.begin()
                    torch.manual_seed(SEED + 5000 * sample_id + 10 * cond_index + (0 if arm == "est" else 1))
                    if torch.cuda.is_available():
                        torch.cuda.manual_seed_all(
                            SEED + 5000 * sample_id + 10 * cond_index + (0 if arm == "est" else 1)
                        )
                    inverted = inverse.invert(
                        prompt=PROMPT,
                        image=undone,
                        num_inference_steps=NUM_INFERENCE_STEPS,
                        guidance_scale=GUIDANCE_SCALE,
                        inpaint_strength=1.0,
                        num_reg_steps=0,
                    )
                    z_hat = _recovered_latent(inverted.latents, z.shape)
                    z_hat = z_hat.detach().to(device="cpu", dtype=torch.float32).contiguous()
                    if tuple(z_hat.shape) != tuple(z.shape) or not torch.isfinite(z_hat).all():
                        raise RuntimeError(f"bad undo latent sample {sample_id} {name}/{arm}")
                    blob[key] = z_hat
                    inv_clock.end(f"sample {sample_id} {name}/{arm}")
            torch.save(blob, path)
            print(
                f"Saved undo latents {path} "
                f"({fmt_duration(time.perf_counter() - sample_started)}).",
                flush=True,
            )
    finally:
        _release_pipeline(inverse)
        print("Image-undo inversion pipeline released.", flush=True)


undo_started = time.perf_counter()
run_image_undo_and_reinvert()
print(f"image-undo wall {fmt_duration(time.perf_counter() - undo_started)}", flush=True)


## 7. Score image-undo bit accuracy vs latent-undo baseline

For each non-baseline trial compare:

| score | meaning |
| --- | --- |
| `raw_warped_*` | signs of warped invert vs baseline / `z` (no undo) |
| `latent_undo_bit_accuracy` | inverse-warp `z_hat` in latent space (prior ~60% finding) |
| `image_undo_est_*` | re-invert after undoing with **estimated** pixel affine |
| `image_undo_oracle_*` | re-invert after undoing with **true** pixel affine |

Bit accuracy is sign agreement on the same circular mask used for registration.



In [ ]:
incomplete_undo = [i for i in range(NUM_SAMPLES) if not undo_pass_complete(i)]
if incomplete_undo:
    raise FileNotFoundError(f"incomplete image-undo samples {incomplete_undo}")

blobs = [torch.load(sample_path(i), map_location="cpu", weights_only=True) for i in range(NUM_SAMPLES)]
rows = []
for sample_id in range(NUM_SAMPLES):
    blob = blobs[sample_id]
    z = as_chw(blob["z"])
    baseline = as_chw(blob[latent_key("baseline")])
    for _, reg in registration.loc[registration.sample_id == sample_id].iterrows():
        name = reg["condition"]
        observation = as_chw(blob[latent_key(name)])
        row = reg.to_dict()
        row["corr_baseline_vs_z"] = full_latent_corr(baseline, z, VALID_MASK)

        if name == "baseline":
            # Identity: image undo is the baseline invert itself.
            row["image_undo_est_bit_accuracy_vs_baseline"] = 1.0
            row["image_undo_oracle_bit_accuracy_vs_baseline"] = 1.0
            row["image_undo_est_bit_accuracy_vs_z"] = sign_bit_accuracy(z, baseline, VALID_MASK)
            row["image_undo_oracle_bit_accuracy_vs_z"] = row["image_undo_est_bit_accuracy_vs_z"]
            row["image_undo_est_corr_vs_z"] = full_latent_corr(baseline, z, VALID_MASK)
            row["image_undo_oracle_corr_vs_z"] = row["image_undo_est_corr_vs_z"]
            row["image_undo_est_corr_vs_baseline"] = 1.0
            row["image_undo_oracle_corr_vs_baseline"] = 1.0
        else:
            for arm in ("est", "oracle"):
                recovered = as_chw(blob[undo_latent_key(name, arm)])
                row[f"image_undo_{arm}_bit_accuracy_vs_baseline"] = sign_bit_accuracy(
                    baseline, recovered, VALID_MASK
                )
                row[f"image_undo_{arm}_bit_accuracy_vs_z"] = sign_bit_accuracy(z, recovered, VALID_MASK)
                row[f"image_undo_{arm}_corr_vs_z"] = full_latent_corr(recovered, z, VALID_MASK)
                row[f"image_undo_{arm}_corr_vs_baseline"] = full_latent_corr(
                    recovered, baseline, VALID_MASK
                )
        rows.append(row)

results = pd.DataFrame(rows)
results.to_csv(OUTPUT_DIR / "trial_results.csv", index=False)
print(f"wrote {OUTPUT_DIR / 'trial_results.csv'} ({len(results)} rows)")

# Quick peek at the comparison that matters.
nonbase = results.loc[results.condition != "baseline"]
print(
    "non-baseline means | "
    f"raw={nonbase.raw_warped_bit_accuracy_vs_baseline.mean():.3f} "
    f"latent_undo={nonbase.latent_undo_bit_accuracy.mean():.3f} "
    f"image_est={nonbase.image_undo_est_bit_accuracy_vs_baseline.mean():.3f} "
    f"image_oracle={nonbase.image_undo_oracle_bit_accuracy_vs_baseline.mean():.3f}"
)


## 8. Summaries and plots



In [ ]:
summary = (
    results.groupby("condition", observed=True)
    .agg(
        n=("sample_id", "count"),
        raw_warped_bit_accuracy_vs_baseline=("raw_warped_bit_accuracy_vs_baseline", "mean"),
        latent_undo_bit_accuracy=("latent_undo_bit_accuracy", "mean"),
        image_undo_est_bit_accuracy_vs_baseline=("image_undo_est_bit_accuracy_vs_baseline", "mean"),
        image_undo_oracle_bit_accuracy_vs_baseline=("image_undo_oracle_bit_accuracy_vs_baseline", "mean"),
        raw_warped_bit_accuracy_vs_z=("raw_warped_bit_accuracy_vs_z", "mean"),
        image_undo_est_bit_accuracy_vs_z=("image_undo_est_bit_accuracy_vs_z", "mean"),
        image_undo_oracle_bit_accuracy_vs_z=("image_undo_oracle_bit_accuracy_vs_z", "mean"),
        image_undo_est_corr_vs_z=("image_undo_est_corr_vs_z", "mean"),
        image_undo_oracle_corr_vs_z=("image_undo_oracle_corr_vs_z", "mean"),
        matched_param_mae=("matched_param_mae", "mean"),
        matched_angle_err=("matched_angle_err", "mean"),
        matched_tx_err=("matched_tx_err", "mean"),
        matched_ty_err=("matched_ty_err", "mean"),
    )
    .reset_index()
)
order = {c["name"]: i for i, c in enumerate(CONDITIONS)}
summary["ord"] = summary.condition.map(order)
summary = summary.sort_values("ord").drop(columns=["ord"]).reset_index(drop=True)
summary.to_csv(OUTPUT_DIR / "summary_by_condition.csv", index=False)
print(summary.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))
x = np.arange(len(summary))
width = 0.2
axes[0].bar(x - 1.5 * width, summary.raw_warped_bit_accuracy_vs_baseline, width=width, label="raw warped")
axes[0].bar(x - 0.5 * width, summary.latent_undo_bit_accuracy, width=width, label="latent undo")
axes[0].bar(x + 0.5 * width, summary.image_undo_est_bit_accuracy_vs_baseline, width=width, label="image undo (est)")
axes[0].bar(x + 1.5 * width, summary.image_undo_oracle_bit_accuracy_vs_baseline, width=width, label="image undo (oracle)")
axes[0].axhline(0.5, color="0.4", ls=":", label="chance")
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary.condition, rotation=25, ha="right")
axes[0].set_ylim(0.45, 1.02)
axes[0].set_ylabel("bit accuracy vs baseline z_hat")
axes[0].set_title("Sign bits vs baseline invert")
axes[0].legend(fontsize=7)

axes[1].bar(x - width, summary.raw_warped_bit_accuracy_vs_z, width=width, label="raw warped")
axes[1].bar(x, summary.image_undo_est_bit_accuracy_vs_z, width=width, label="image undo (est)")
axes[1].bar(x + width, summary.image_undo_oracle_bit_accuracy_vs_z, width=width, label="image undo (oracle)")
axes[1].axhline(0.5, color="0.4", ls=":", label="chance")
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary.condition, rotation=25, ha="right")
axes[1].set_ylim(0.45, 1.02)
axes[1].set_ylabel("bit accuracy vs original z")
axes[1].set_title("Sign bits vs Gaussian watermark z")
axes[1].legend(fontsize=7)
fig.suptitle(f"Image-undo re-invert vs latent undo ({NUM_SAMPLES} samples)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "bit_accuracy_by_undo_method.png", bbox_inches="tight")
plt.show()

nonbase = results.loc[results.condition != "baseline"]
fig, ax = plt.subplots(figsize=(5.5, 5.0))
ax.scatter(nonbase.latent_undo_bit_accuracy, nonbase.image_undo_est_bit_accuracy_vs_baseline, s=45)
lim = (0.45, 1.02)
ax.plot(lim, lim, ls="--", color="0.35", label="y = x")
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel("latent inverse-warp bit accuracy")
ax.set_ylabel("image-undo (est) bit accuracy vs baseline")
ax.set_title("Does pixel undo + re-invert beat latent undo?")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "image_undo_vs_latent_undo_scatter.png", bbox_inches="tight")
plt.show()

# Visual: warped vs estimated undo vs oracle undo for sample 0 / first nonbaseline.
demo_cond = NONBASELINE[0]["name"]
demo_dir = sample_image_dir(0)
panel_paths = [
    (f"{demo_cond}.png", "warped"),
    (f"{demo_cond}__undo_est.png", "undo est"),
    (f"{demo_cond}__undo_oracle.png", "undo oracle"),
    ("baseline.png", "baseline"),
]
fig, axes = plt.subplots(1, 4, figsize=(11, 2.9))
for ax, (fname, title) in zip(axes, panel_paths):
    path = demo_dir / fname
    if path.is_file():
        ax.imshow(Image.open(path))
    ax.set_title(title, fontsize=9)
    ax.axis("off")
fig.suptitle(f"Sample 0 / {demo_cond}: image undo panels")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "example_image_undo_panels_sample0.png", bbox_inches="tight")
plt.show()


## 9. Interpretation



In [ ]:
from experiments.generate_and_invert import _git_commit

meta_path = OUTPUT_DIR / "metadata.json"
meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.is_file() else {}
meta.update(
    {
        "experiment": "estimate latent affine → inverse-warp warped image → re-invert → bit accuracy",
        "registration": "sign-only bits of baseline z_hat vs full condition z_hat",
        "latent_undo_baseline": "bit_metrics inverse-warp of z_hat (prior ~60% finding)",
        "image_undo_arms": ["est", "oracle"],
        "expected_latent_map": "same linear params; translation_px / vae_scale_factor",
        "n_samples": NUM_SAMPLES,
        "trial_rows": int(len(results)),
        "reuse_from": str(REUSE_FROM),
        "git_commit": meta.get("git_commit") or _git_commit(),
    }
)
meta_path.write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")

print("Interpretation")
base = results.loc[results.condition == "baseline"]
print(
    f"- Baseline latent_undo bit={base.latent_undo_bit_accuracy.mean():.3f}; "
    f"image_undo est vs baseline={base.image_undo_est_bit_accuracy_vs_baseline.mean():.3f}."
)
print(
    f"- Non-baseline raw warped vs baseline={nonbase.raw_warped_bit_accuracy_vs_baseline.mean():.3f}."
)
print(
    f"- Non-baseline latent undo (prior finding)={nonbase.latent_undo_bit_accuracy.mean():.3f}."
)
print(
    f"- Non-baseline image undo est vs baseline="
    f"{nonbase.image_undo_est_bit_accuracy_vs_baseline.mean():.3f}; "
    f"oracle={nonbase.image_undo_oracle_bit_accuracy_vs_baseline.mean():.3f}."
)
print(
    f"- Non-baseline image undo est vs z="
    f"{nonbase.image_undo_est_bit_accuracy_vs_z.mean():.3f}; "
    f"oracle={nonbase.image_undo_oracle_bit_accuracy_vs_z.mean():.3f}."
)
gain_est = (
    nonbase.image_undo_est_bit_accuracy_vs_baseline.mean()
    - nonbase.latent_undo_bit_accuracy.mean()
)
gain_oracle = (
    nonbase.image_undo_oracle_bit_accuracy_vs_baseline.mean()
    - nonbase.latent_undo_bit_accuracy.mean()
)
print(f"- Image-est minus latent-undo gap: {gain_est:.3f}.")
print(f"- Image-oracle minus latent-undo gap: {gain_oracle:.3f}.")
for name in [c["name"] for c in NONBASELINE]:
    block = results.loc[results.condition == name]
    print(
        f"- {name}: latent={block.latent_undo_bit_accuracy.mean():.3f} "
        f"image_est={block.image_undo_est_bit_accuracy_vs_baseline.mean():.3f} "
        f"oracle={block.image_undo_oracle_bit_accuracy_vs_baseline.mean():.3f} | "
        f"ang/tx/ty err="
        f"{block.matched_angle_err.mean():.3f}/"
        f"{block.matched_tx_err.mean():.3f}/"
        f"{block.matched_ty_err.mean():.3f}."
    )
print(f"- Artifacts under {OUTPUT_DIR}")
print(f"- Total wall since start ~{fmt_duration(time.perf_counter() - run_started)}")
